# 02: Exploratory data analysis (B.4)

Goal: understand the target well enough to define it in the README (B.5), and collect feature ideas
for week 3. Every section answers a question that changes a decision.

Uses the **training period only** (2024-01 .. 2025-06), the 20 stations and the service types from
`config.toml`. Needs `make download` and `make process`.

Data: Rijden de Treinen (CC BY 4.0).

## Setup
Settings come from the project config, so this notebook can't disagree with the pipeline.

In [ ]:
import time
from pathlib import Path

import duckdb

from ns_delay.config import get_config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET_GLOB = str(ROOT / "data" / "processed" / "services" / "*.parquet")
# Rijden de Treinen stations list: Dutch stations only, so a code not in it = a station abroad.
STATIONS_CSV = str(ROOT / "data" / "raw" / "stations-2023-09-nl.csv")

config = get_config()
STATIONS = list(config.scope.stations)  # DuckDB turns a Python list into a LIST parameter
SERVICE_TYPES = list(config.scope.service_types)
TRAIN_END = config.split.valid_start  # exclusive: training data is everything before this date
THRESHOLD = config.target.delay_threshold_minutes

con = duckdb.connect()  # in-memory database, lives as long as the kernel
# Show and group timestamps in Dutch local time, explicitly: the default is the machine's time
# zone, which would be UTC on a cloud server and shift every "hour of day" by 1-2 hours.
con.sql("SET TimeZone = 'Europe/Amsterdam'")

## Step 0: one table of in-scope departures

`deps` = one row per departure at one of our 20 stations, in the training period, on a **domestic**
trip.

The trip-level columns (`stop_seq`, `n_stops`, `crosses_border`) are computed over the **whole trip
first** and only then filtered to our stations; otherwise a train from Amersfoort through Utrecht
would look like it started at Utrecht, and a train to Brussels would look domestic. They come from
the timetable, so they are known before departure (no leakage).

Three things found while building this:
- `stop_id` is **not** always in route order: a stop added later (e.g. a diversion) gets a higher id
  but sits in the middle of the trip. So stops are ordered by **planned time**.
- Cancelled departures have `departure_delay = 0`, not NULL (checked below), so `late` is set to
  NULL for them by hand.
- The service-type allowlist does **not** remove international trains: NS International, Eurobahn,
  Arriva etc. use ordinary type names (`Intercity`, `Stoptrein`). So trips with any stop outside
  the Netherlands are excluded explicitly (found in 4.4).

Two objects: `deps_all` keeps the cross-border trips (with the `crosses_border` flag) so 4.4 can
show the evidence; `deps` is a view on it without them, and is what every section uses.

In [ ]:
start = time.perf_counter()
con.execute(
    f"""
    CREATE OR REPLACE TABLE deps_all AS
    WITH trips AS (
        SELECT
            p.service_id, p.service_date, p.service_type, p.company, p.train_number,
            p.service_completely_cancelled, p.service_partly_cancelled,
            p.station_code, p.arrival_time, p.arrival_delay, p.arrival_cancelled,
            p.departure_time, p.departure_delay, p.departure_cancelled, p.platform_change,
            row_number() OVER trip AS stop_seq,
            count(*) OVER (PARTITION BY p.service_id) AS n_stops,
            -- true if any stop of the trip is not a Dutch station (no match in the stations file)
            bool_or(nl.code IS NULL) OVER (PARTITION BY p.service_id) AS crosses_border
        FROM read_parquet($glob) AS p
        LEFT JOIN read_csv($stations_csv) AS nl ON p.station_code = nl.code
        WHERE p.service_date < $train_end
          AND list_contains($service_types, p.service_type)
        WINDOW trip AS (
            PARTITION BY p.service_id
            ORDER BY coalesce(p.arrival_time, p.departure_time), p.stop_id
        )
    )
    SELECT
        *,
        stop_seq = 1 AS is_origin,
        -- NULL for cancelled: the archive stores their delay as 0, which is unknown, not on time.
        -- Target option A (section 2): cancelled departures are not part of the target.
        CASE WHEN NOT departure_cancelled THEN departure_delay >= {THRESHOLD} END AS late
    FROM trips
    WHERE list_contains($stations, station_code)
      AND departure_time IS NOT NULL
    """,
    {
        "glob": PARQUET_GLOB,
        "stations_csv": STATIONS_CSV,
        "train_end": TRAIN_END,
        "service_types": SERVICE_TYPES,
        "stations": STATIONS,
    },
)
# A view has no parameters, so it can be a view: no second copy of the data in memory.
con.sql("""
    CREATE OR REPLACE VIEW deps AS
    SELECT * EXCLUDE (crosses_border) FROM deps_all WHERE NOT crosses_border
""")
print(f"Built deps_all and deps in {time.perf_counter() - start:.0f}s")
con.sql("""
    SELECT crosses_border, count(*) AS departures
    FROM deps_all
    GROUP BY ALL
    ORDER BY ALL
""").show()

### Sanity checks
Expected from notebook 01: ~8,500 departures/day and ~6.8% cancelled, both a bit lower now that
international and cross-border trips are out. Notebook 01's 5.8% late rate included international
trains, so it should come out clearly lower here.

In [3]:
con.sql("""
    SELECT
        count(*) AS departures,
        count(DISTINCT service_date) AS days,
        round(count(*) / count(DISTINCT service_date)) AS per_day,
        count(DISTINCT station_code) AS stations,
        min(service_date) AS first_day,
        max(service_date) AS last_day,
        round(100 * avg(departure_cancelled::INT), 2) AS pct_cancelled,
        round(100 * avg(late::INT), 2) AS pct_late,  -- avg skips NULLs, so cancelled are left out
        round(100 * avg((departure_delay >= 5)::INT), 2) AS pct_late_naive  -- cancelled as on time
    FROM deps
""").show()

print("What delay do cancelled departures carry? (unknown should not look like 'on time')")
con.sql("""
    SELECT
        departure_cancelled,
        count(*) AS n,
        count(*) FILTER (WHERE departure_delay IS NULL) AS delay_null,
        min(departure_delay) AS min_delay,
        max(departure_delay) AS max_delay
    FROM deps
    GROUP BY ALL
    ORDER BY ALL
""").show()

┌────────────┬───────┬─────────┬──────────┬────────────┬────────────┬───────────────┬──────────┬────────────────┐
│ departures │ days  │ per_day │ stations │ first_day  │  last_day  │ pct_cancelled │ pct_late │ pct_late_naive │
│   int64    │ int64 │ double  │  int64   │    date    │    date    │    double     │  double  │     double     │
├────────────┼───────┼─────────┼──────────┼────────────┼────────────┼───────────────┼──────────┼────────────────┤
│    4555306 │   547 │  8328.0 │       20 │ 2024-01-01 │ 2025-06-30 │          6.75 │     5.31 │           4.96 │
└────────────┴───────┴─────────┴──────────┴────────────┴────────────┴───────────────┴──────────┴────────────────┘

What delay do cancelled departures carry? (unknown should not look like 'on time')
┌─────────────────────┬─────────┬────────────┬───────────┬───────────┐
│ departure_cancelled │    n    │ delay_null │ min_delay │ max_delay │
│       boolean       │  int64  │   int64    │   int32   │   int32   │
├──────────────────

### What step 0 tells us
- **4,449,168 departures** in 547 days (~8,100/day), all 20 stations present, no day missing.
  106,138 more departures belong to cross-border trips and are left out (`deps_all` keeps them).
- **Late >=5 min: 4.94%** of the trains that ran: imbalanced (~1 late per 20), so accuracy is useless
  and PR-AUC / Brier are the right metrics. Lower than notebook 01's 5.8% because international and
  cross-border trains are now excluded.
- **Cancelled: 6.53%**, more common than late.
- **Cancelled departures have `departure_delay = 0`**, never NULL. A naive `delay >= 5` target
  calls them on time (4.62% instead of 4.94%). `late` is NULL for them (target option A, section 2).
- **`stop_id` is not always in route order** (~0.3% of trips), so stop order uses planned time.

## Section 1: what does the delay look like?

Questions:
1. How is `departure_delay` distributed, and how sensitive is the positive rate to the threshold?
2. What is in the tail: real long delays, or data errors?
3. Does the distribution differ by service type?

Cancelled departures are left out everywhere here: their delay of 0 means "unknown".

### 1.1 Distribution and threshold sensitivity
`pct` = share of departures with exactly this delay; `pct_at_least` = share with at least this delay,
i.e. the positive rate if this were the threshold. Delays of 16+ minutes are grouped into one row.

In [4]:
con.sql("""
    WITH counts AS (
        SELECT least(departure_delay, 16) AS delay_min, count(*) AS n
        FROM deps
        WHERE NOT departure_cancelled
        GROUP BY ALL
    )
    SELECT
        delay_min,
        n,
        round(100 * n / sum(n) OVER (), 2) AS pct,
        round(100 * sum(n) OVER (ORDER BY delay_min DESC) / sum(n) OVER (), 2) AS pct_at_least,
        bar(100 * n / sum(n) OVER (), 0, 75, 40) AS histogram
    FROM counts
    ORDER BY delay_min
""").show(max_rows=20)

┌───────────┬─────────┬────────┬──────────────┬──────────────────────────────────────────┐
│ delay_min │    n    │  pct   │ pct_at_least │                histogram                 │
│   int32   │  int64  │ double │    double    │                 varchar                  │
├───────────┼─────────┼────────┼──────────────┼──────────────────────────────────────────┤
│         0 │ 3118322 │  73.41 │        100.0 │ ███████████████████████████████████████▏ │
│         1 │  490156 │  11.54 │        26.59 │ ██████▏                                  │
│         2 │  215905 │   5.08 │        15.05 │ ██▋                                      │
│         3 │  122351 │   2.88 │         9.97 │ █▌                                       │
│         4 │   75280 │   1.77 │         7.09 │ ▉                                        │
│         5 │   51195 │   1.21 │         5.31 │ ▋                                        │
│         6 │   35548 │   0.84 │         4.11 │ ▍                                        │

### 1.2 The tail
Quantiles, very long delays, and rows whose planned time is on the day *before* the service date
(after-midnight trains are on the day after, which is normal).

In [5]:
con.sql("""
    SELECT
        round(avg(departure_delay), 2) AS mean,
        median(departure_delay) AS p50,
        quantile_disc(departure_delay, 0.90) AS p90,
        quantile_disc(departure_delay, 0.95) AS p95,
        quantile_disc(departure_delay, 0.99) AS p99,
        quantile_disc(departure_delay, 0.999) AS p999,
        max(departure_delay) AS max,
        count(*) FILTER (WHERE departure_delay > 60) AS over_60,
        count(*) FILTER (WHERE departure_delay > 180) AS over_180,
        round(avg(departure_delay) FILTER (WHERE late), 1) AS mean_if_late,
        median(departure_delay) FILTER (WHERE late) AS median_if_late
    FROM deps
    WHERE NOT departure_cancelled
""").show()

# day_offset -1 = planned the day *before* the service date: suspicious.
print("Planned departure date minus service date:")
con.sql("""
    SELECT
        departure_time::DATE - service_date AS day_offset,
        count(*) AS n,
        max(departure_delay) AS max_delay
    FROM deps
    GROUP BY ALL
    ORDER BY ALL
""").show()

con.sql("""
    SELECT
        service_date, station_code, train_number, departure_time::VARCHAR AS planned,
        departure_delay, departure_time + to_minutes(departure_delay) AS actual
    FROM deps
    WHERE departure_time::DATE < service_date
    ORDER BY departure_delay DESC
""").show()

┌────────┬────────┬───────┬───────┬───────┬───────┬───────┬─────────┬──────────┬──────────────┬────────────────┐
│  mean  │  p50   │  p90  │  p95  │  p99  │ p999  │  max  │ over_60 │ over_180 │ mean_if_late │ median_if_late │
│ double │ double │ int32 │ int32 │ int32 │ int32 │ int32 │  int64  │  int64   │    double    │     double     │
├────────┼────────┼───────┼───────┼───────┼───────┼───────┼─────────┼──────────┼──────────────┼────────────────┤
│   0.91 │    0.0 │     2 │     5 │    13 │    32 │  1440 │     858 │        9 │         10.0 │            8.0 │
└────────┴────────┴───────┴───────┴───────┴───────┴───────┴─────────┴──────────┴──────────────┴────────────────┘

Planned departure date minus service date:
┌────────────┬─────────┬───────────┐
│ day_offset │    n    │ max_delay │
│   int64    │  int64  │   int32   │
├────────────┼─────────┼───────────┤
│         -1 │      13 │      1440 │
│          0 │ 4433897 │       306 │
│          1 │  121396 │       120 │
└────────────┴─────

### 1.3 By service type

In [6]:
con.sql("""
    SELECT
        service_type,
        count(*) AS departures,
        round(100 * count(*) / sum(count(*)) OVER (), 1) AS pct_of_all,
        round(100 * avg(late::INT), 2) AS pct_late,
        round(100 * avg(departure_cancelled::INT), 2) AS pct_cancelled,
        round(100 * avg((departure_delay = 0)::INT) FILTER (WHERE NOT departure_cancelled), 1)
            AS pct_zero_delay
    FROM deps
    GROUP BY ALL
    ORDER BY departures DESC
""").show()

┌──────────────────┬────────────┬────────────┬──────────┬───────────────┬────────────────┐
│   service_type   │ departures │ pct_of_all │ pct_late │ pct_cancelled │ pct_zero_delay │
│     varchar      │   int64    │   double   │  double  │    double     │     double     │
├──────────────────┼────────────┼────────────┼──────────┼───────────────┼────────────────┤
│ Intercity        │    2144402 │       47.1 │     6.27 │          8.65 │           70.4 │
│ Sprinter         │    1843566 │       40.5 │     3.86 │          4.18 │           76.9 │
│ Stoptrein        │     298425 │        6.6 │     2.07 │          3.52 │           83.3 │
│ Intercity direct │     212327 │        4.7 │     14.8 │         12.23 │           54.1 │
│ Sneltrein        │      56586 │        1.2 │     2.55 │         15.07 │           80.3 │
└──────────────────┴────────────┴────────────┴──────────┴───────────────┴────────────────┘



### 1.4 What the archive cannot tell us: rounding and early departures
- The archive has **whole minutes and no negative delays** (minimum 0), so early departures are
  stored as 0 and we can't see how seconds were rounded.
- The live API gives exact times; our parser floors them (5:59 -> 5) and keeps early ones as
  negative (-30 s -> -1).
- To avoid training/serving skew, serving must reproduce the archive's convention: **clip
  negative delays to 0**. Whether the archive floors or rounds can only be checked by comparing
  live API data with the archive for the same trains (possible once live polling runs in week 2).
  At the 5-minute boundary this matters: 1.2% of departures are exactly 5 min late.

### What section 1 tells us
- **Most departures are on time**: 74.0% have delay 0, 85% are at most 1 minute late.
  The distribution is extremely skewed (mean 0.8, median 0, p99 12 min).
- **The positive rate falls steeply with the threshold**: >=3 min 9.5%, >=5 min 4.9%,
  >=10 min 1.6%. 5 minutes keeps enough positives (~205k in training) and matches what a
  traveller notices. (NS has used both 3- and 5-minute punctuality definitions: check the current
  one before citing it in the README.)
  Every minute around 5 is worth ~1 percentage point of positives, so rounding matters (1.4).
- **When late, typically 7 min** (median; mean 9.2). That's the scale for the agent's answers.
- **The tail is short once international trains are out**: 128 delays over an hour (858 with them),
  the longest believable one 306 min.
  **But 13 rows have a planned time on the day before the service date**, all from a handful
  of trains on 3 days. 4 of them show delays of ~1440 min = exactly 24 h: the planned date is
  wrong by one day, not the train. -> Data-quality rule (week 2): drop rows whose planned
  departure date is before the service date.
- **Service type matters a lot**: Intercity direct 11.4% late, Intercity 5.9%, Sprinter 3.9%,
  Stoptrein 1.7%, Sneltrein 0.9%. Strong feature.
- **Archive limits**: no negative delays and whole minutes only, so serving must clip early
  departures to 0; floor vs round can only be checked against live data (week 2).

## Section 2: how do cancellations show up?

Cancellations (6.75%) are more common than late departures (5.31%), so how they enter the target is
the biggest open decision (B.5). Questions:
1. How do the three cancellation flags relate?
2. What does a cancellation look like at one stop?
3. Are cancellations spread evenly, or concentrated on a few days?
4. What would each target definition mean for the positive rate?

### 2.1 The three flags
`departure_cancelled` is about this stop; the two `service_*` flags describe the **whole trip**,
including stops after this one (so they are future information, like `Maximum delay`).

In [7]:
con.sql("""
    SELECT
        service_completely_cancelled AS trip_fully,
        service_partly_cancelled AS trip_partly,
        departure_cancelled AS this_departure,
        count(*) AS n,
        round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM deps
    GROUP BY ALL
    ORDER BY ALL
""").show()

┌────────────┬─────────────┬────────────────┬─────────┬────────┐
│ trip_fully │ trip_partly │ this_departure │    n    │  pct   │
│  boolean   │   boolean   │    boolean     │  int64  │ double │
├────────────┼─────────────┼────────────────┼─────────┼────────┤
│ false      │ false       │ false          │ 4008725 │   88.0 │
│ false      │ false       │ true           │       1 │    0.0 │
│ false      │ true        │ false          │  238602 │   5.24 │
│ false      │ true        │ true           │  218928 │   4.81 │
│ true       │ false       │ false          │     402 │   0.01 │
│ true       │ false       │ true           │       1 │    0.0 │
│ true       │ true        │ false          │      51 │    0.0 │
│ true       │ true        │ true           │   88596 │   1.94 │
└────────────┴─────────────┴────────────────┴─────────┴────────┘



### 2.2 One stop: arrival vs departure cancelled
Origins have no arrival (`NULL`). For through stations, the combination says what happened here:
both cancelled = the train skipped the stop; only one = the trip was cut short or started here.

In [8]:
con.sql("""
    SELECT
        is_origin,
        arrival_cancelled,
        departure_cancelled,
        CASE
            WHEN is_origin THEN 'origin'
            WHEN arrival_cancelled AND departure_cancelled THEN 'skipped this stop'
            WHEN departure_cancelled THEN 'arrived, but trip ends here (cut short)'
            WHEN arrival_cancelled THEN 'did not arrive, but starts here'
            ELSE 'normal'
        END AS meaning,
        count(*) AS n
    FROM deps
    GROUP BY ALL
    ORDER BY is_origin, n DESC
""").show(max_width=150)

┌───────────┬───────────────────┬─────────────────────┬─────────────────────────────────────────┬─────────┐
│ is_origin │ arrival_cancelled │ departure_cancelled │                 meaning                 │    n    │
│  boolean  │      boolean      │       boolean       │                 varchar                 │  int64  │
├───────────┼───────────────────┼─────────────────────┼─────────────────────────────────────────┼─────────┤
│ false     │ false             │ false               │ normal                                  │ 2933404 │
│ false     │ true              │ true                │ skipped this stop                       │  202632 │
│ false     │ false             │ true                │ arrived, but trip ends here (cut short) │   39866 │
│ false     │ true              │ false               │ did not arrive, but starts here         │   39708 │
│ false     │ NULL              │ false               │ normal                                  │     169 │
│ false     │ NULL          

### 2.3 Are cancellations concentrated in time?
One row per day in `daily`. Then: the month view, the worst days, and how much of all cancellations
the worst 5% of days hold.

In [9]:
con.sql("""
    CREATE OR REPLACE TABLE daily AS
    SELECT
        service_date,
        count(*) AS departures,
        count(*) FILTER (WHERE departure_cancelled) AS cancelled,
        count(late) AS ran,  -- count(column) skips NULLs, and late is NULL when cancelled
        count(*) FILTER (WHERE late) AS late,
        avg(departure_cancelled::INT) AS pct_cancelled
    FROM deps
    GROUP BY ALL
""")

print("By month:")
con.sql("""
    SELECT
        strftime(service_date, '%Y-%m') AS month,
        round(100 * sum(cancelled) / sum(departures), 2) AS pct_cancelled,
        round(100 * sum(late) / sum(ran), 2) AS pct_late
    FROM daily
    GROUP BY ALL
    ORDER BY ALL
""").show(max_rows=20)

print("Worst 10 days:")
con.sql("""
    SELECT
        service_date,
        dayname(service_date) AS weekday,
        departures,
        round(100 * pct_cancelled, 1) AS pct_cancelled,
        round(100 * late / ran, 1) AS pct_late
    FROM daily
    ORDER BY pct_cancelled DESC
    LIMIT 10
""").show()

By month:
┌─────────┬───────────────┬──────────┐
│  month  │ pct_cancelled │ pct_late │
│ varchar │    double     │  double  │
├─────────┼───────────────┼──────────┤
│ 2024-01 │          7.76 │     7.36 │
│ 2024-02 │          5.53 │     6.65 │
│ 2024-03 │          5.57 │     5.61 │
│ 2024-04 │          5.55 │     4.79 │
│ 2024-05 │          5.99 │     5.94 │
│ 2024-06 │          5.75 │     5.25 │
│ 2024-07 │          5.89 │     5.07 │
│ 2024-08 │          5.28 │     4.69 │
│ 2024-09 │          5.75 │     5.96 │
│ 2024-10 │          4.78 │     5.58 │
│ 2024-11 │          7.89 │     7.92 │
│ 2024-12 │          5.54 │     4.61 │
│ 2025-01 │          8.97 │     4.57 │
│ 2025-02 │           6.7 │     4.67 │
│ 2025-03 │          6.25 │      4.4 │
│ 2025-04 │          4.96 │     4.22 │
│ 2025-05 │          5.65 │     4.06 │
│ 2025-06 │         17.76 │     4.37 │
└─────────┴───────────────┴──────────┘
  18 rows                  3 columns

Worst 10 days:
┌──────────────┬───────────┬────────────

In [10]:
# Split the days into 20 equal groups by cancellation rate (group 1 = the worst 5% of days).
con.sql("""
    WITH ranked AS (
        SELECT *, ntile(20) OVER (ORDER BY pct_cancelled DESC) AS group_20
        FROM daily
    )
    SELECT
        group_20,
        count(*) AS days,
        round(100 * min(pct_cancelled), 1) AS min_pct_cancelled,
        round(100 * sum(cancelled) / (SELECT sum(cancelled) FROM daily), 1) AS share_of_cancelled,
        round(100 * sum(late) / sum(ran), 2) AS pct_late
    FROM ranked
    GROUP BY ALL
    ORDER BY group_20
""").show(max_rows=20)

con.sql("""
    SELECT round(corr(pct_cancelled, late / ran), 2) AS corr_cancelled_vs_late_per_day
    FROM daily
""").show()

┌──────────┬───────┬───────────────────┬────────────────────┬──────────┐
│ group_20 │ days  │ min_pct_cancelled │ share_of_cancelled │ pct_late │
│  int64   │ int64 │      double       │       double       │  double  │
├──────────┼───────┼───────────────────┼────────────────────┼──────────┤
│        1 │    28 │              13.8 │               20.6 │     6.96 │
│        2 │    28 │              10.6 │                9.8 │      7.6 │
│        3 │    28 │               9.1 │                7.8 │     5.78 │
│        4 │    28 │               8.3 │                7.0 │     5.96 │
│        5 │    28 │               7.5 │                6.1 │     6.18 │
│        6 │    28 │               7.0 │                5.6 │     5.54 │
│        7 │    28 │               6.5 │                5.2 │     5.52 │
│        8 │    27 │               6.0 │                4.7 │     5.81 │
│        9 │    27 │               5.7 │                4.4 │     5.41 │
│       10 │    27 │               5.2 │           

### 2.4 What each target definition would mean
- **A: late, among trains that run**: cancelled departures are left out. "If my train runs, will it be late?"
- **B: late or cancelled**: one "bad departure" label. "Will my departure go wrong?"

In [11]:
con.sql("""
    SELECT
        round(100 * avg(late::INT), 2) AS a_pct_positive,
        count(late) AS a_rows,
        round(100 * avg((late OR departure_cancelled)::INT), 2) AS b_pct_positive,
        count(*) AS b_rows,
        round(100 * avg((late OR departure_cancelled)::INT)
            FILTER (WHERE strftime(service_date, '%Y-%m') <> '2025-06'), 2) AS b_without_june_2025
    FROM deps
""").show()

┌────────────────┬─────────┬────────────────┬─────────┬─────────────────────┐
│ a_pct_positive │ a_rows  │ b_pct_positive │ b_rows  │ b_without_june_2025 │
│     double     │  int64  │     double     │  int64  │       double        │
├────────────────┼─────────┼────────────────┼─────────┼─────────────────────┤
│           5.31 │ 4247780 │          11.71 │ 4555306 │               11.15 │
└────────────────┴─────────┴────────────────┴─────────┴─────────────────────┘



### What section 2 tells us
- **The trip-level flags are future information.** ~10% of departures belong to a partly cancelled
  trip, but in half of them (5.19%) *this* departure ran: the cancellation happened elsewhere on the
  trip, often later. `service_partly_cancelled` / `service_completely_cancelled` must never be
  features (add them to the leakage test with `Maximum delay`). Only `departure_cancelled` is about
  this departure.
- **Most stop-level cancellations are skipped stops** (195k), plus ~37k trips cut short here and
  ~37k that started here instead. 59k cancelled departures are at the origin.
- **Cancellations are concentrated in a few days.** The worst 5% of days (28 days) hold 21% of all
  cancellations; June 2025 alone is 17.6% cancelled, with two days above 80% (6 and 10 June).
  These look like announced events (strikes, planned works) rather than everyday disruption:
  to check before writing it in the README.
- **On those days, the trains that do run are not unusually late** (6 June: 87% cancelled,
  2.3% late). Day-level correlation between cancellations and lateness is only 0.19; the
  high-cancellation days are somewhat later (~6-7.5% vs ~3.5%), so bad days do exist, but
  cancellations and delays are mostly different processes.
- **Target options:** A (late among trains that run) = 4.94% positive, 4.16M rows.
  B (late or cancelled) = 11.15% positive, more than half of the positives would be cancellations,
  and a big part of them from a handful of event days.
- **The archive can't tell when a cancellation became known.** A strike announced days ahead is
  already "cancelled" in the live API at prediction time; there is nothing to predict.

**Decision for B.5: option A.** The target is "departure delay >= 5 min, given that the train runs";
cancelled departures are excluded from training and evaluation. Cancellations are handled by the
agent (live API: is it cancelled?) and the disruption model (how long will it last?). Possible
stretch goal: a separate cancellation model.

## Section 3: when are trains late?

Questions:
1. How does the late rate change over the day, and is it different at the weekend?
2. Which weekdays are worse?
3. Seasons and drift: is the same month later in 2024 than in 2025?
4. Are bad days isolated, or do they come in runs?

All rates are among trains that ran (`late` is NULL for cancelled, and `avg` skips NULLs).
Hours are Amsterdam time (set in the setup cell).

### 3.1 Hour of day, weekday vs weekend
`isodow` = ISO day of week, Monday = 1 .. Sunday = 7.

In [12]:
con.sql("""
    SELECT
        hour(departure_time) AS hour,
        count(late) AS ran,
        round(100 * avg(late::INT) FILTER (WHERE isodow(service_date) <= 5), 2) AS pct_late_weekday,
        round(100 * avg(late::INT) FILTER (WHERE isodow(service_date) >= 6), 2) AS pct_late_weekend,
        bar(100 * avg(late::INT), 0, 10, 30) AS all_days
    FROM deps
    GROUP BY ALL
    ORDER BY hour
""").show(max_rows=24)

┌───────┬────────┬──────────────────┬──────────────────┬────────────────────────────────┐
│ hour  │  ran   │ pct_late_weekday │ pct_late_weekend │            all_days            │
│ int64 │ int64  │      double      │      double      │            varchar             │
├───────┼────────┼──────────────────┼──────────────────┼────────────────────────────────┤
│     0 │  98055 │             7.49 │             7.36 │ ██████████████████████▎        │
│     1 │  16335 │             9.81 │             8.77 │ ████████████████████████████▏  │
│     2 │   5606 │             9.56 │             8.06 │ ██████████████████████████▉    │
│     3 │   4559 │             7.41 │             6.77 │ █████████████████████▌         │
│     4 │   5887 │             4.83 │             5.69 │ ███████████████▏               │
│     5 │  59736 │             2.65 │             3.34 │ ████████                       │
│     6 │ 168808 │             2.92 │             2.36 │ ████████▌                      │
│     7 │ 

### 3.2 Day of week
`per_day` = average departures that ran on that weekday (fewer trains at the weekend).

In [13]:
con.sql("""
    SELECT
        isodow(service_date) AS day_nr,
        dayname(service_date) AS weekday,
        round(count(late) / count(DISTINCT service_date)) AS per_day,
        round(100 * avg(late::INT), 2) AS pct_late
    FROM deps
    GROUP BY ALL
    ORDER BY day_nr
""").show()

┌────────┬───────────┬─────────┬──────────┐
│ day_nr │  weekday  │ per_day │ pct_late │
│ int64  │  varchar  │ double  │  double  │
├────────┼───────────┼─────────┼──────────┤
│      1 │ Monday    │  8286.0 │      5.4 │
│      2 │ Tuesday   │  8319.0 │     5.12 │
│      3 │ Wednesday │  8377.0 │      5.2 │
│      4 │ Thursday  │  8390.0 │     6.13 │
│      5 │ Friday    │  7870.0 │     5.85 │
│      6 │ Saturday  │  6807.0 │     4.86 │
│      7 │ Sunday    │  6303.0 │     4.37 │
└────────┴───────────┴─────────┴──────────┘



### 3.3 Month of year, 2024 vs 2025
The training period has 18 months: Jan-Jun twice, Jul-Dec once. Comparing the same month across
years separates **season** (both years high) from **drift** (one year lower everywhere).

In [14]:
con.sql("""
    SELECT
        month(service_date) AS month,
        round(100 * avg(late::INT) FILTER (WHERE year(service_date) = 2024), 2) AS pct_late_2024,
        round(100 * avg(late::INT) FILTER (WHERE year(service_date) = 2025), 2) AS pct_late_2025
    FROM deps
    GROUP BY ALL
    ORDER BY month
""").show(max_rows=12)

print("Same half-year, both years:")
con.sql("""
    SELECT
        year(service_date) AS year,
        round(100 * avg(late::INT), 2) AS pct_late_jan_jun
    FROM deps
    WHERE month(service_date) <= 6
    GROUP BY ALL
    ORDER BY year
""").show()

┌───────┬───────────────┬───────────────┐
│ month │ pct_late_2024 │ pct_late_2025 │
│ int64 │    double     │    double     │
├───────┼───────────────┼───────────────┤
│     1 │          7.36 │          4.57 │
│     2 │          6.65 │          4.67 │
│     3 │          5.61 │           4.4 │
│     4 │          4.79 │          4.22 │
│     5 │          5.94 │          4.06 │
│     6 │          5.25 │          4.37 │
│     7 │          5.07 │          NULL │
│     8 │          4.69 │          NULL │
│     9 │          5.96 │          NULL │
│    10 │          5.58 │          NULL │
│    11 │          7.92 │          NULL │
│    12 │          4.61 │          NULL │
└───────┴───────────────┴───────────────┘
  12 rows                     3 columns

Same half-year, both years:
┌───────┬──────────────────┐
│ year  │ pct_late_jan_jun │
│ int64 │      double      │
├───────┼──────────────────┤
│  2024 │             5.93 │
│  2025 │             4.38 │
└───────┴──────────────────┘



### 3.4 Bad days: isolated, or in runs?
Late rate per day, then the worst days, then the correlation between a day and the day before
(`lag` = the value from the previous row in the window's order).

In [15]:
con.sql("""
    CREATE OR REPLACE TABLE daily_late AS
    SELECT service_date, count(*) FILTER (WHERE late) / count(late) AS pct_late
    FROM deps
    GROUP BY ALL
""")

con.sql("""
    SELECT
        round(100 * median(pct_late), 1) AS median_day,
        round(100 * quantile_disc(pct_late, 0.9), 1) AS p90_day,
        round(100 * quantile_disc(pct_late, 0.99), 1) AS p99_day,
        round(100 * max(pct_late), 1) AS worst_day
    FROM daily_late
""").show()

print("Worst 10 days:")
con.sql("""
    SELECT service_date, dayname(service_date) AS weekday, round(100 * pct_late, 1) AS pct_late
    FROM daily_late
    ORDER BY pct_late DESC
    LIMIT 10
""").show()

con.sql("""
    WITH pairs AS (
        SELECT pct_late, lag(pct_late) OVER (ORDER BY service_date) AS pct_late_day_before
        FROM daily_late
    )
    SELECT round(corr(pct_late, pct_late_day_before), 2) AS corr_with_day_before
    FROM pairs
""").show()

┌────────────┬─────────┬─────────┬───────────┐
│ median_day │ p90_day │ p99_day │ worst_day │
│   double   │ double  │ double  │  double   │
├────────────┼─────────┼─────────┼───────────┤
│        5.0 │     7.6 │    12.2 │      19.5 │
└────────────┴─────────┴─────────┴───────────┘

Worst 10 days:
┌──────────────┬───────────┬──────────┐
│ service_date │  weekday  │ pct_late │
│     date     │  varchar  │  double  │
├──────────────┼───────────┼──────────┤
│ 2024-11-27   │ Wednesday │     19.5 │
│ 2024-11-25   │ Monday    │     17.3 │
│ 2024-01-22   │ Monday    │     14.6 │
│ 2024-01-19   │ Friday    │     14.2 │
│ 2024-11-22   │ Friday    │     13.2 │
│ 2024-01-11   │ Thursday  │     12.2 │
│ 2024-11-28   │ Thursday  │     11.8 │
│ 2024-01-23   │ Tuesday   │     11.3 │
│ 2024-06-05   │ Wednesday │     11.0 │
│ 2024-09-27   │ Friday    │     10.8 │
└──────────────┴───────────┴──────────┘
  10 rows                   3 columns

┌──────────────────────┐
│ corr_with_day_before │
│        doub

### What section 3 tells us
- **Delay builds up during the day.** Lowest at 05:00-07:00 (~2.5-3.6%: trains start fresh from
  the depot), rising to ~6% in the evening; 00:00-03:00 is highest (7-10%) but has few trains.
- **The weekday evening peak is the worst time** (17:00-19:00 weekday ~6.3-6.5%). At the weekend
  there is no morning peak (08:00: 5.3% weekday vs 3.0% weekend). -> Features: hour,
  weekday/weekend, and their interaction (a tree model like LightGBM learns interactions by itself).
- **Thursday is the worst day (5.7%), Sunday the best (4.1%).**
- **Strong drift between years:** Jan-Jun 2024 5.48% vs Jan-Jun 2025 4.18%, and *every* month of
  2025 is lower than the same month of 2024. Consequences:
  - The model trained on all of 2024-mid 2025 will likely **over-predict** in the validation
    period: calibration should be fitted on the validation period, and checked on test.
  - **Month of year is unreliable as a feature:** Jul-Dec appears only once (2024), so "November"
    would mean "November 2024" (7.4%, the storm-heavy month), not "any November".
  - Worth trying in week 4: give recent months more weight, or a shorter training window.
- **Bad days come in runs:** the worst days are clustered (22-28 Nov 2024, 11-23 Jan 2024), and a
  day's late rate correlates 0.44 with the day before. Something network-wide drives them
  (weather? to check with KNMI in week 2). -> The strongest feature idea: **the current state of the
  network** (e.g. share of late departures in the last hour), known at prediction time.

## Section 4: where are trains late?

Questions:
1. Do trains that start at a station leave on time more often than trains passing through?
2. Does lateness grow along the trip?
3. How much of a station's late rate is explained by how many trains start there?

`is_origin`, `stop_seq` and `n_stops` were computed in step 0 over the whole trip, from the
timetable, so they are known before departure.

### 4.1 Origin vs through, per service type

In [ ]:
con.sql("""
    SELECT
        coalesce(service_type, 'ALL') AS service_type,  -- ROLLUP's total row has NULL here
        count(late) FILTER (WHERE is_origin) AS ran_origin,
        round(100 * avg(late::INT) FILTER (WHERE is_origin), 2) AS pct_late_origin,
        count(late) FILTER (WHERE NOT is_origin) AS ran_through,
        round(100 * avg(late::INT) FILTER (WHERE NOT is_origin), 2) AS pct_late_through
    FROM deps
    GROUP BY ROLLUP (service_type)
    ORDER BY ran_origin + ran_through DESC
""").show()

### 4.2 Position in the trip
`stop_seq` alone mixes short and long trips (stop 10 is the end of one trip, the middle of another),
so use the **relative position**: 0.0 = origin, 0.9 = the last stretch before the terminus.
(Termini have no departure, so they are not in `deps`.)

In [ ]:
con.sql("""
    SELECT
        floor(10 * (stop_seq - 1) / (n_stops - 1)) / 10 AS position,
        count(late) AS ran,
        round(100 * avg(late::INT), 2) AS pct_late,
        bar(100 * avg(late::INT), 0, 10, 30) AS bar
    FROM deps
    WHERE n_stops > 1
    GROUP BY ALL
    ORDER BY position
""").show()

### 4.3 Per station: is it the station, or how many trains start there?

In [ ]:
con.sql("""
    SELECT
        station_code,
        count(late) AS ran,
        round(100 * avg(is_origin::INT), 1) AS pct_starting_here,
        round(100 * avg(late::INT), 2) AS pct_late,
        round(100 * avg(late::INT) FILTER (WHERE is_origin), 2) AS pct_late_origin,
        round(100 * avg(late::INT) FILTER (WHERE NOT is_origin), 2) AS pct_late_through
    FROM deps
    GROUP BY ALL
    ORDER BY pct_late DESC
""").show(max_rows=20)

### What section 4 tells us
- **Where the train starts matters most.** A departure from the origin is 2.7% late, a departure
  further along the trip 5.9%, and this holds for every service type (Intercity direct: 5.8% vs
  13.8%). Trains start clean; delay is picked up on the way.
- **Lateness grows along the trip:** 2.7% at the origin, ~4-5.5% in the first half, 6.4-7.7% in the
  second half, a bit lower on the last stretch. -> Features: `is_origin` and the relative position
  `(stop_seq - 1) / (n_stops - 1)`, both from the timetable. (The 20 rows at position 1.0 are
  departures from the last stop: odd, ignore.)
- **A station's late rate is partly composition.** ZL is 3.9% overall but 6.2% for trains passing
  through: it looks punctual because 55% of its trains start there. Same for ASD (51% start there,
  7.4% late when passing through). Comparing stations without `is_origin` is misleading; the model
  should get both the station and `is_origin`.
- **Real station effects remain:** BD (8.9% through) and SHL (7.8%) stay bad; UT (4.3%) and
  AH (3.3%) stay good. -> `station_code` is a useful feature on top of the position.
- **HGL was the exception** (origin later than through) before cross-border trips were removed;
  4.4 explains why. Now HGL is 3.1% at the origin vs 3.4% through.

### 4.4 Why are trains *starting* at Hengelo late?
HGL broke the pattern of 4.1: 11.0% late at the origin vs 5.3% passing through (before the
cross-border trips were removed in step 0, which is the result of this check). These cells use
`deps_all`, which still has them.

**Step 1: which trains?** Group by series (`train_number // 100`): trains of one line share a series.

In [ ]:
con.sql("""
    SELECT
        service_type,
        train_number // 100 AS series,
        count(late) AS ran,
        count(*) FILTER (WHERE late) AS n_late,
        round(100 * avg(late::INT), 2) AS pct_late
    FROM deps_all  -- deps no longer has the cross-border trains
    WHERE station_code = 'HGL' AND is_origin
    GROUP BY ALL
    ORDER BY ran DESC
    LIMIT 8
""").show()

**Step 2: who runs series 203, and where does it go?** `deps` has neither `company` nor stations
outside our 20, so read the Parquet files directly. `TRAIN_END` keeps it in the training period.

In [ ]:
con.sql(
    """
    SELECT company, count(DISTINCT service_id) AS trips
    FROM read_parquet($glob)
    WHERE train_number // 100 = 203 AND service_date < $train_end
    GROUP BY ALL
    """,
    params={"glob": PARQUET_GLOB, "train_end": TRAIN_END},
).show()

# One trip that ends at Hengelo, all its stops in time order.
con.sql(
    """
    WITH one_trip AS (
        SELECT service_id
        FROM read_parquet($glob)
        WHERE train_number // 100 = 203 AND station_code = 'HGL' AND departure_time IS NULL
          AND service_date = DATE '2024-03-05'
        LIMIT 1
    )
    SELECT
        train_number, station_code, station_name,
        arrival_time::VARCHAR AS arrival, departure_time::VARCHAR AS departure,
        arrival_delay, departure_delay
    FROM read_parquet($glob)
    WHERE service_id = (SELECT service_id FROM one_trip)
    ORDER BY coalesce(arrival_time, departure_time)
    """,
    params={"glob": PARQUET_GLOB},
).show(max_width=200)

**Step 3: where was the train just before it "started"?** The archive doesn't say which physical
train runs which trip, so we infer it: for every departure of series 203 from HGL, take the **last
arrival of series 203 at HGL before it** on the same day (an `ASOF JOIN`). Then: the planned
turnaround time, and the late rate by how late the incoming train was.

In [ ]:
con.execute(
    """
    CREATE OR REPLACE TABLE hgl_turnaround AS
    WITH inbound AS (  -- series 203 trips ending at HGL (no departure there)
        SELECT service_date, train_number, arrival_time, arrival_delay
        FROM read_parquet($glob)
        WHERE station_code = 'HGL' AND train_number // 100 = 203
          AND departure_time IS NULL AND NOT arrival_cancelled
          AND service_date < $train_end
    ),
    outbound AS (  -- series 203 trips starting at HGL
        SELECT service_date, train_number, departure_time, departure_delay, late
        FROM deps_all
        WHERE station_code = 'HGL' AND train_number // 100 = 203 AND is_origin AND late IS NOT NULL
    )
    SELECT
        o.*,
        i.train_number AS inbound_train,
        i.arrival_delay AS inbound_delay,
        date_diff('minute', i.arrival_time, o.departure_time) AS planned_turnaround_min
    FROM outbound AS o
    ASOF LEFT JOIN inbound AS i
        ON o.service_date = i.service_date AND i.arrival_time <= o.departure_time
    """,
    {"glob": PARQUET_GLOB, "train_end": TRAIN_END},
)

con.sql("""
    SELECT
        count(*) AS departures,
        count(inbound_train) AS with_inbound,
        median(planned_turnaround_min) AS median_turnaround_min,
        round(corr(inbound_delay, departure_delay), 2) AS corr_inbound_vs_departure
    FROM hgl_turnaround
""").show()

con.sql("""
    SELECT
        CASE
            WHEN inbound_delay IS NULL THEN 'no inbound found'
            WHEN inbound_delay < 5 THEN '0-4 min'
            WHEN inbound_delay < 10 THEN '5-9 min'
            ELSE '10+ min'
        END AS inbound_delay,
        count(*) AS departures,
        round(100 * avg(late::INT), 1) AS pct_late
    FROM hgl_turnaround
    GROUP BY ALL
    ORDER BY ALL
""").show()

**Step 4: is this only Hengelo?** Series 203 is a cross-border train labelled `Stoptrein`, so the
service-type allowlist didn't remove it. Which departures belong to a trip that also stops
**outside the Netherlands**? The stations file lists Dutch stations only, so "a stop whose code is
not in the file" = abroad. Step 0 now flags these trips (`crosses_border`) and leaves them out of
`deps`; `deps_all` still has them, for this comparison.

In [ ]:
# deps_all has the crosses_border flag from step 0 (same rule as explained above).
print("Cross-border departures, by type and company:")
con.sql("""
    SELECT
        service_type,
        company,
        count(late) AS ran,
        round(100 * avg(late::INT), 2) AS pct_late,
        string_agg(DISTINCT station_code, ',' ORDER BY station_code) AS our_stations
    FROM deps_all
    WHERE crosses_border
    GROUP BY ALL
    ORDER BY ran DESC
""").show(max_width=200)

print("What it does to the totals and to each station:")
con.sql("""
    SELECT
        coalesce(station_code, 'ALL') AS station_code,
        count(late) FILTER (WHERE crosses_border) AS ran_cross_border,
        round(100 * avg(late::INT), 2) AS pct_late_with,
        round(100 * avg(late::INT) FILTER (WHERE NOT crosses_border), 2) AS pct_late_domestic
    FROM deps_all
    GROUP BY ROLLUP (station_code)
    HAVING ran_cross_border > 0
    ORDER BY ran_cross_border DESC
""").show(max_rows=20)

print("Every station code treated as 'abroad' (check for Dutch stations newer than the file):")
con.sql(
    """
    SELECT station_code, any_value(station_name) AS name, count(*) AS stops
    FROM read_parquet($glob)
    WHERE service_date < $train_end
      AND station_code NOT IN (SELECT code FROM read_csv($stations_csv))
    GROUP BY ALL
    ORDER BY stops
    LIMIT 10
    """,
    params={"glob": PARQUET_GLOB, "train_end": TRAIN_END, "stations_csv": STATIONS_CSV},
).show()

### What the HGL check tells us
- **Series 203 is a Eurobahn train between Bielefeld (Germany) and Hengelo**, labelled `Stoptrein`.
  It is ~8,000 of HGL's origin departures, 13.3% late.
- **"Origin" doesn't mean "fresh" here.** The train arrives from Germany and leaves again after a
  planned **8 minutes**. If it came in 0-4 min late, the next trip is ~1% late; 5-9 min: ~31%;
  10+ min: ~97%. The delay is carried over by the vehicle, not by the trip.
  -> Feature idea (week 3): *the delay of the incoming train at a terminus*. Careful: it's only known
  at prediction time if that train has already arrived, or as its delay *so far* (leakage test).
- **The bigger finding: the allowlist does not remove international trains.** ~89k departures
  (2.1%) in `deps` belong to cross-border trips, and they are **22.8% late** vs 4.9% for domestic
  trips. The big ones are NS International's `Intercity direct` (Amsterdam-Brussels) and
  `Intercity` (Amsterdam-Berlin) at 31-35% late, both at our biggest stations.
- It inflates exactly the stations we called "bad": BD 10.2% -> 8.2% domestic, HGL 5.7% -> 3.4%,
  DV 7.3% -> 5.5%, RTD 7.1% -> 6.1%. The overall rate drops from 5.31% to 4.94%.
- The station-file method works: every code it flags is abroad, except one Rotterdam steam depot
  (`RTNG`, 1 row).

**Decision: exclude every trip with a stop outside the Netherlands** (step 0 now does this).
It matches the reason for leaving international trains out (their delay is caused abroad) and
catches every operator, not just NS International. In the pipeline it belongs in the scope filter
(week 3 dataset build, with a config switch), not in `process.py`, which keeps all raw data. At
serving time the same rule needs the train's route stations from the NS API.